# PCS x C2C: does the cache bridge transfer *beliefs*, or just help?

**Question.** Cache-to-Cache (C2C, ICLR 2026, arXiv 2510.03215) trains a small fuser that maps a frozen *sharer* LLM's KV cache into a frozen *receiver* LLM's KV cache. Papers like it score task accuracy. PCS asks something different: when the sharer is **wrong**, does the receiver inherit the sharer's wrong answer (the bridge carries the sharer's cognition) or does it override it (the bridge is a generic helper)?

**Pair.** receiver `Qwen/Qwen3-0.6B`, sharer `Qwen/Qwen2.5-0.5B-Instruct`, released fuser `nics-efc/C2C_Fuser` / `qwen3_0.6b+qwen2.5_0.5b_Fuser/final` (about 956 MB). C2C code is cloned and pinned to a fixed commit.

**Read `PREREG.md` first.** It fixes hypotheses, sample sizes, metrics and decision thresholds before any result exists.

| Part | What | Items | Approx. T4 time |
|---|---|---|---|
| A | plumbing check vs the paper's OpenBookQA numbers (generation readout and logit readout) | 200 | 5-8 min |
| B | **fidelity test**: inheritance rate on sharer/receiver disagreements, with text-handoff and mismatched-sharer controls | 2000 | 25-40 min |
| C | planted false belief only the sharer sees (exploratory) | 500 | 5-10 min |

Setup (installs, model download) takes about 8-12 min. Everything is greedy and seeded. Outputs: `results.json` (per-item predictions for every condition plus summary), written to `/kaggle/working` on Kaggle or the current directory elsewhere.

**Run it:** Kaggle: Settings > Accelerator > **GPU T4**, Internet **on**, then Run All. Colab: Runtime > T4 GPU, Run all. For a ~10 minute rehearsal with small sizes (not for results) set `QUICK = True` in the config cell or env `PCS_QUICK=1`.

In [ ]:
# --- 1. Config ---
import os, sys, json, time, math, random, re, subprocess, platform, tempfile, hashlib, datetime, gc
from pathlib import Path

def _flag(name):
    return os.environ.get(name, "").lower() in ("1", "true", "yes")

SMOKE = _flag("PCS_SMOKE")   # CPU rehearsal with tiny RANDOM models (author's validation only; results are meaningless)
QUICK = _flag("PCS_QUICK")   # True = small sizes on a real GPU (rehearsal only; never use for results)

CFG = dict(
    SEED=0,
    # sample sizes: pre-registered in PREREG.md
    N_PLUMB=200, N_MMLU=1000, N_ARC=600, N_OBQA=400, N_RESERVE_MMLU=1000, MIN_CELL=150, N_PLANT=500,
    MAX_TOKENS=1536,            # drop items whose prompt is longer than this (C2C trained with max 2048)
    MMLU_SUBJECT_LIMIT=None,    # None = all 57 subjects
    RUN_A=True, RUN_B=True, RUN_C=True,
    RUN_GEN_READOUT_IN_A=True,  # paper protocol (generate 64 tokens)
    RUN_T2_RATIONALE=True,      # text handoff that also carries the sharer's one-sentence rationale
    DTYPE="auto",               # auto | bf16 | fp16 | fp32
    PIN_TORCH=False,            # True: install torch==2.6.0 (cu124), the version C2C pins. Default keeps the platform's torch.
    TIME_BUDGET_MIN=90,         # soft cap for parts B+C; N is shrunk (and logged) if the estimate exceeds it
    N_BOOT=10000,
)
if QUICK or SMOKE:
    CFG.update(N_PLUMB=12 if SMOKE else 40, N_MMLU=24 if SMOKE else 100, N_ARC=16 if SMOKE else 60,
               N_OBQA=12 if SMOKE else 40, N_RESERVE_MMLU=0, MIN_CELL=1, N_PLANT=12 if SMOKE else 40,
               MMLU_SUBJECT_LIMIT=6, N_BOOT=500)

SEED = CFG["SEED"]
IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or Path("/content").exists())
OUT_DIR = Path(os.environ.get("PCS_OUT", "/kaggle/working" if IS_KAGGLE else str(Path.cwd())))
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = OUT_DIR / "results.json"
PARTIAL_PATH = OUT_DIR / "results_partial.json"

RECV_ID = "Qwen/Qwen3-0.6B"
SHAR_ID = "Qwen/Qwen2.5-0.5B-Instruct"
FUSER_REPO = "nics-efc/C2C_Fuser"
FUSER_SUB = "qwen3_0.6b+qwen2.5_0.5b_Fuser"
C2C_URL = "https://github.com/thu-nics/C2C"
C2C_COMMIT = "3ca0e98020bac1f36000623afcb8dfc6f8c29bdf"   # code version this notebook was audited against
PINS = {"transformers": "4.52.4", "tokenizers": "0.21.4", "huggingface_hub": "0.34.3"}  # C2C's own pins

DEVIATIONS = []   # anything that departs from PREREG.md is appended here and saved to results.json
T0 = time.time()
print(f"env: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'local'} | python {platform.python_version()} | out={OUT_DIR}")
print("mode:", "SMOKE (tiny random models, CPU)" if SMOKE else "QUICK (small sizes)" if QUICK else "FULL")

In [ ]:
# --- 2. Install pinned deps, GPU sanity (runs BEFORE torch is imported) ---
def sh(cmd, check=True):
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check)

def gpu_query():
    try:
        o = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=20)
        if o.returncode == 0 and o.stdout.strip():
            name, cc, mem = [x.strip() for x in o.stdout.strip().splitlines()[0].split(",")]
            return name, float(cc), mem
    except Exception:
        pass
    return None

def installed(pkg):
    import importlib.metadata as md
    try:
        return md.version(pkg)
    except Exception:
        return None

if "torch" in sys.modules and not SMOKE:
    print("note: torch is already imported in this kernel; if installs below change anything, restart and Run All")

if not SMOKE:
    g = gpu_query()
    if g is None:
        raise RuntimeError("No NVIDIA GPU found. Kaggle: Settings > Accelerator > GPU T4. Colab: Runtime > Change runtime type > T4 GPU.")
    print("GPU:", g)
    if (g[1] < 7.0 or CFG["PIN_TORCH"]) and "torch" not in sys.modules:
        # e.g. Kaggle P100 (sm_60): recent default torch wheels may lack sm_60 kernels. C2C itself pins torch 2.6.0.
        sh([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
    need = [f"{k}=={v}" for k, v in PINS.items() if installed(k) != v]
    if need:
        sh([sys.executable, "-m", "pip", "install", "-q", *need])
    for pkg in ("datasets", "pandas", "numpy", "tqdm"):
        if installed(pkg) is None:
            sh([sys.executable, "-m", "pip", "install", "-q", pkg])

for k, v in PINS.items():
    got = installed(k)
    if got != v:
        raise RuntimeError(f"{k}=={got}, expected {v}. Restart the kernel and Run All (the install cell must run before anything imports {k}).")
print({k: installed(k) for k in list(PINS) + ["datasets", "numpy"]})

In [ ]:
# --- 3. Clone C2C at the audited commit ---
C2C_DIR = Path(os.environ.get("PCS_C2C_DIR", str(Path(tempfile.gettempdir()) / "C2C")))
if not (C2C_DIR / "rosetta").exists():
    C2C_DIR.mkdir(parents=True, exist_ok=True)
    try:
        sh(["git", "init", "-q", str(C2C_DIR)])
        sh(["git", "-C", str(C2C_DIR), "remote", "add", "origin", C2C_URL])
        sh(["git", "-C", str(C2C_DIR), "fetch", "-q", "--depth", "1", "origin", C2C_COMMIT])
        sh(["git", "-C", str(C2C_DIR), "checkout", "-q", "FETCH_HEAD"])
    except Exception as e:
        print("pinned checkout failed (", e, ") -> falling back to default branch HEAD")
        DEVIATIONS.append("C2C pinned commit unavailable; used HEAD of default branch")
        __import__("shutil").rmtree(C2C_DIR, ignore_errors=True)
        sh(["git", "clone", "-q", "--depth", "1", C2C_URL, str(C2C_DIR)])
sys.path.insert(0, str(C2C_DIR))   # `rosetta` is a namespace package: do NOT `pip install -e .` (it would re-pin torch)
head = subprocess.run(["git", "-C", str(C2C_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("C2C commit:", head)

In [ ]:
# --- 4. Imports, device, dtype policy, API presence checks ---
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, GenerationConfig
from huggingface_hub import snapshot_download

from rosetta.model.wrapper import RosettaModel
from rosetta.model.projector import load_projector
from rosetta.utils.evaluate import build_prompt, get_option_token_ids, extract_answer_from_content, set_default_chat_template

for _n in ("forward", "generate", "load_projector_config"):
    assert hasattr(RosettaModel, _n), f"C2C API changed: RosettaModel.{_n} missing"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    GPU_NAME = torch.cuda.get_device_name(0)
    CC = torch.cuda.get_device_capability(0)
    NATIVE_BF16 = CC[0] >= 8
    torch.cuda.manual_seed_all(SEED)
else:
    if not SMOKE:
        raise RuntimeError("CUDA not available. Enable a GPU runtime.")
    DEVICE, GPU_NAME, CC, NATIVE_BF16 = torch.device("cpu"), "cpu", (0, 0), False
print(f"torch {torch.__version__} | transformers {installed('transformers')} | device {DEVICE} ({GPU_NAME}, cc {CC}) | native bf16: {NATIVE_BF16}")
LETTERS = "ABCD"
RESP_TEXT = "The correct answer is"   # C2C's default `response_text` for answer_method=logits
GEN_TOKENS = 8 if SMOKE else 64        # paper protocol: max 64 new tokens

In [ ]:
# --- 5. Data: three MC benchmarks, C2C's own prompt builder and answer parsing ---
from datasets import load_dataset

MMLU_SUBJECTS = [
    'abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge', 'college_biology',
    'college_chemistry', 'college_computer_science', 'college_mathematics', 'college_medicine', 'college_physics',
    'computer_security', 'conceptual_physics', 'econometrics', 'electrical_engineering', 'elementary_mathematics',
    'formal_logic', 'global_facts', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science',
    'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics',
    'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics', 'high_school_physics',
    'high_school_psychology', 'high_school_statistics', 'high_school_us_history', 'high_school_world_history',
    'human_aging', 'human_sexuality', 'international_law', 'jurisprudence', 'logical_fallacies', 'machine_learning',
    'management', 'marketing', 'medical_genetics', 'miscellaneous', 'moral_disputes', 'moral_scenarios', 'nutrition',
    'philosophy', 'prehistory', 'professional_accounting', 'professional_law', 'professional_medicine',
    'professional_psychology', 'public_relations', 'security_studies', 'sociology', 'us_foreign_policy', 'virology',
    'world_religions']
if CFG["MMLU_SUBJECT_LIMIT"]:
    MMLU_SUBJECTS = MMLU_SUBJECTS[::max(1, len(MMLU_SUBJECTS) // CFG["MMLU_SUBJECT_LIMIT"])][:CFG["MMLU_SUBJECT_LIMIT"]]

def retry(fn, tries=4):
    for k in range(tries):
        try:
            return fn()
        except Exception as e:
            if k == tries - 1:
                raise
            print(f"  retry {k+1} after error: {str(e)[:120]}")
            time.sleep(3 * (k + 1))

def make_prompt(question, choices):
    choice_str = "".join(f"{LETTERS[i]}. {c}\n" for i, c in enumerate(choices))
    return build_prompt(dataset="mmlu-redux", locale="", question=question, choices=choice_str, use_cot=False, use_template=True)

def mmlu_redux_key(ex):
    """Mirror of C2C UnifiedEvaluator.parse_answer for mmlu-redux (skips no_correct_answer / expert)."""
    et = ex.get("error_type", "") or ""
    if et in ("no_correct_answer", "expert"):
        return None
    if et == "wrong_groundtruth" and ex.get("correct_answer") not in (None, ""):
        ca = str(ex["correct_answer"]).strip()
        if ca in LETTERS:
            return ca
        if ca in ("0", "1", "2", "3"):
            return LETTERS[int(ca)]
        return None
    a = int(ex["answer"])
    return LETTERS[a] if 0 <= a <= 3 else None

def load_pools():
    pools = {}
    rows = []
    for ex in retry(lambda: load_dataset("allenai/openbookqa", "main", split="test")):
        ch = list(ex["choices"]["text"])
        if len(ch) == 4 and ex["answerKey"] in LETTERS:
            rows.append(dict(bench="obqa", uid=ex["id"], question=ex["question_stem"], choices=ch, y=ex["answerKey"]))
    pools["obqa"] = rows
    rows = []
    for ex in retry(lambda: load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")):
        ch, lab = list(ex["choices"]["text"]), list(ex["choices"]["label"])
        if len(ch) == 4 and lab == list(LETTERS) and ex["answerKey"] in LETTERS:   # C2C also only keeps A-D keys
            rows.append(dict(bench="arc", uid=ex["id"], question=ex["question"], choices=ch, y=ex["answerKey"]))
    pools["arc"] = rows
    rows = []
    for subj in MMLU_SUBJECTS:
        for i, ex in enumerate(retry(lambda: load_dataset("edinburgh-dawg/mmlu-redux-2.0", subj, split="test"))):
            y = mmlu_redux_key(ex)
            if y is not None and len(ex["choices"]) == 4:
                rows.append(dict(bench="mmlu", uid=f"{subj}:{i}", question=ex["question"], choices=list(ex["choices"]), y=y, subject=subj))
    pools["mmlu"] = rows
    return pools

t = time.time()
POOLS = load_pools()
print({k: len(v) for k, v in POOLS.items()}, f"loaded in {time.time()-t:.0f}s")

In [ ]:
# --- 6. Models + fuser (dtype-aware loader; the repo's own loader hard-codes bf16, which a T4 cannot do natively) ---
def fuser_dir():
    pats = [f"{FUSER_SUB}/final/*.json"] if SMOKE else [f"{FUSER_SUB}/final/*", f"{FUSER_SUB}/config.json"]
    root = snapshot_download(repo_id=FUSER_REPO, allow_patterns=pats)
    return os.path.join(root, FUSER_SUB, "final")

FUSER_PATH = fuser_dir()
print("fuser dir:", FUSER_PATH, "| files:", len(os.listdir(FUSER_PATH)))

def _tiny(cfg_id, **over):
    cfg = AutoConfig.from_pretrained(cfg_id)
    for k, v in over.items():
        setattr(cfg, k, v)
    return cfg

def build_stack(dtype):
    tok_r = AutoTokenizer.from_pretrained(RECV_ID); set_default_chat_template(tok_r, RECV_ID)
    tok_s = AutoTokenizer.from_pretrained(SHAR_ID)
    if SMOKE:
        torch.manual_seed(SEED)
        recv = AutoModelForCausalLM.from_config(_tiny(RECV_ID, hidden_size=128, intermediate_size=256, num_attention_heads=16,
                                                      num_key_value_heads=8, head_dim=128, num_hidden_layers=28), torch_dtype=dtype)
        shar = AutoModelForCausalLM.from_config(_tiny(SHAR_ID, hidden_size=128, intermediate_size=256, num_attention_heads=2,
                                                      num_key_value_heads=2, num_hidden_layers=24), torch_dtype=dtype)
    else:
        recv = AutoModelForCausalLM.from_pretrained(RECV_ID, torch_dtype=dtype)
        shar = AutoModelForCausalLM.from_pretrained(SHAR_ID, torch_dtype=dtype)
    recv, shar = recv.to(DEVICE).eval(), shar.to(DEVICE).eval()
    n_proj = len([f for f in os.listdir(FUSER_PATH) if re.fullmatch(r"projector_\d+\.json", f)])
    projs = []
    for t_ in range(n_proj):
        over = {"dtype": dtype}
        if SMOKE:
            over.update(hidden_dim=64, intermediate_dim=64)
        p = load_projector(os.path.join(FUSER_PATH, f"projector_{t_}.json"), override_args=over)
        pt = os.path.join(FUSER_PATH, f"projector_{t_}.pt")
        if not SMOKE:
            res = p.load_state_dict(torch.load(pt, map_location="cpu", weights_only=True), strict=False)   # repo uses strict=False; we check
            if res.missing_keys or res.unexpected_keys:
                raise RuntimeError(f"fuser projector_{t_}: missing={res.missing_keys} unexpected={res.unexpected_keys}")
        if SMOKE:   # random projectors start with closed gates (logit 0 -> no fusion); open them so the fusion path is exercised
            with torch.no_grad():
                p.key_gate_logit.fill_(1.0); p.value_gate_logit.fill_(1.0)
        projs.append(p.to(DEVICE))
    ros = RosettaModel(model_list=[recv, shar], base_model_idx=0, projector_list=projs).to(DEVICE).eval()
    ros.load_projector_config(os.path.join(FUSER_PATH, "projector_config.json"))
    assert len(ros.projector_dict[0][1]) == recv.config.num_hidden_layers, "projector map must cover every receiver layer"
    return dict(tok_r=tok_r, tok_s=tok_s, recv=recv, shar=shar, ros=ros, dtype=dtype, n_proj=n_proj)

def use_stack(st):
    global TOK_R, TOK_S, RECV, SHAR, ROS, DTYPE_USED, OPT_IDS, OPT_T, RESP_IDS, EOS_R, EOS_S, PAD_R, PAD_S
    TOK_R, TOK_S, RECV, SHAR, ROS, DTYPE_USED = st["tok_r"], st["tok_s"], st["recv"], st["shar"], st["ros"], st["dtype"]
    o_r, o_s = get_option_token_ids(TOK_R, 4), get_option_token_ids(TOK_S, 4)
    assert o_r == o_s, "receiver/sharer tokenizers disagree on option-letter ids"
    OPT_IDS = o_r
    OPT_T = torch.tensor(OPT_IDS, device=DEVICE)
    r1, r2 = TOK_R(RESP_TEXT, add_special_tokens=False)["input_ids"], TOK_S(RESP_TEXT, add_special_tokens=False)["input_ids"]
    assert r1 == r2, "response-suffix tokenization differs between tokenizers"
    RESP_IDS = r1
    EOS_R, EOS_S = RECV.generation_config.eos_token_id, SHAR.generation_config.eos_token_id
    PAD_R = TOK_R.pad_token_id if TOK_R.pad_token_id is not None else (EOS_R[0] if isinstance(EOS_R, list) else EOS_R)
    PAD_S = TOK_S.pad_token_id if TOK_S.pad_token_id is not None else (EOS_S[0] if isinstance(EOS_S, list) else EOS_S)

def free_stack(st):
    for k in list(st):
        st[k] = None
    for n_ in ("RECV", "SHAR", "ROS"):
        globals()[n_] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# --- 7. Inference primitives (all conditions share one readout: letter logits after "The correct answer is") ---
COUNTERS = {"nonfinite": 0}

def encode_user(tok, user_text):
    s = tok.apply_chat_template([{"role": "user", "content": user_text}], tokenize=False,
                                add_generation_prompt=True, enable_thinking=False)
    return tok(s, add_special_tokens=False)["input_ids"]

def to_pred(logits_vec):
    sel = logits_vec[OPT_T].float()
    if not torch.isfinite(sel).all():
        COUNTERS["nonfinite"] += 1
        return {"a": "?", "p": [None] * 4}
    p = torch.softmax(sel, 0).cpu().numpy()
    return {"a": LETTERS[int(p.argmax())], "p": [round(float(x), 4) for x in p]}

@torch.no_grad()
def lm_logits(model, ids, mask=None, positions=None):
    """Plain HF forward (receiver-alone / sharer-alone). Returns last-position logits."""
    x = torch.tensor([ids], device=DEVICE)
    pos = torch.arange(x.shape[1], device=DEVICE).unsqueeze(0) if positions is None else torch.tensor([positions], device=DEVICE)
    am = None if mask is None else torch.tensor([mask], device=DEVICE)
    return model(input_ids=x, attention_mask=am, position_ids=pos, use_cache=False, logits_to_keep=1).logits[0, -1]

def _kv_index(n_instr, n_total, sharer_mask=1):
    a = torch.tensor([sharer_mask, 0], dtype=torch.long).repeat(n_instr, 1).unsqueeze(0).to(DEVICE)
    b = torch.tensor([-1, 0], dtype=torch.long).repeat(n_total - n_instr, 1).unsqueeze(0).to(DEVICE)
    return [a, b]

@torch.no_grad()
def c2c_logits(ids_r, ids_s, n_instr, mask_r=None, mask_s=None, sharer_mask=1, as_tensor=False):
    """C2C-fused forward. ids_r / ids_s: receiver / sharer token ids, same length. The first n_instr positions are
    fused (kv_cache_index [1,0]); the trailing response tokens are not (and the sharer never sees them: C2C only
    computes sharer caches for the fused sections)."""
    assert len(ids_r) == len(ids_s), "C2C needs position-aligned inputs (equal length)"
    n = len(ids_r)
    xr, xs = torch.tensor([ids_r], device=DEVICE), torch.tensor([ids_s], device=DEVICE)
    mr = torch.tensor([mask_r if mask_r is not None else [1] * n], device=DEVICE)
    ms = torch.tensor([mask_s if mask_s is not None else [1] * n], device=DEVICE)
    pos = torch.arange(n, device=DEVICE).unsqueeze(0)
    kv = _kv_index(n_instr, n, sharer_mask)
    if as_tensor:   # exactly the call pattern of C2C's own evaluator for same-tokenizer pairs
        out = ROS.forward(kv_cache_index=kv, input_ids=xr, attention_mask=mr, position_ids=pos)
    else:
        out = ROS.forward(kv_cache_index=kv, input_ids=[xr, xs], attention_mask=[mr, ms], position_ids=pos, use_cache=True)
    return out.logits[0, -1]

def _gen_cfg(n_new, eos, pad):
    return GenerationConfig(do_sample=False, max_new_tokens=n_new, pad_token_id=pad, eos_token_id=eos)

@torch.no_grad()
def hf_generate(model, tok, ids, n_new, eos, pad):
    x = torch.tensor([ids], device=DEVICE)
    out = model.generate(input_ids=x, attention_mask=torch.ones_like(x), generation_config=_gen_cfg(n_new, eos, pad))
    return tok.decode(out[0, x.shape[1]:], skip_special_tokens=True).strip("\n")

@torch.no_grad()
def c2c_generate(ids, n_new=64):
    """README / evaluator pattern: fuse all prompt tokens except the last (response_length=1), greedy decode."""
    x = torch.tensor([ids], device=DEVICE)
    n = x.shape[1]
    pos = torch.arange(n, device=DEVICE).unsqueeze(0)
    out = ROS.generate(kv_cache_index=_kv_index(n - 1, n), input_ids=x, attention_mask=torch.ones_like(x),
                       position_ids=pos, do_sample=False, max_new_tokens=n_new)
    return TOK_R.decode(out[0, n:], skip_special_tokens=True).strip("\n")

@torch.no_grad()
def hf_generate_batch(model, tok, id_lists, n_new, eos, pad, bs=16):
    outs = []
    for i in range(0, len(id_lists), bs):
        chunk = id_lists[i:i + bs]
        m = max(len(c) for c in chunk)
        x = torch.tensor([[pad] * (m - len(c)) + c for c in chunk], device=DEVICE)
        am = torch.tensor([[0] * (m - len(c)) + [1] * len(c) for c in chunk], device=DEVICE)
        out = model.generate(input_ids=x, attention_mask=am, generation_config=_gen_cfg(n_new, eos, pad))
        outs += [tok.decode(o[m:], skip_special_tokens=True) for o in out]
    return outs

In [ ]:
# --- 8. Statistics helpers (bootstrap over items) ---
def boot_ci(x, n_boot=None, seed=0):
    """Percentile bootstrap CI of the mean. Pass per-item differences for paired comparisons."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n == 0:
        return (float("nan"),) * 3
    n_boot = n_boot or CFG["N_BOOT"]
    rng = np.random.default_rng(seed)
    chunk = max(1, int(2_000_000 // n))
    means, done = [], 0
    while done < n_boot:
        m = min(chunk, n_boot - done)
        means.append(x[rng.integers(0, n, size=(m, n))].mean(1))
        done += m
    means = np.concatenate(means)
    return float(x.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

def boot_ci_unpaired_diff(xa, xb, n_boot=None, seed=0):
    xa, xb = np.asarray(xa, float), np.asarray(xb, float)
    if len(xa) == 0 or len(xb) == 0:
        return (float("nan"),) * 3
    n_boot = n_boot or CFG["N_BOOT"]
    rng = np.random.default_rng(seed)
    da = np.array([xa[rng.integers(0, len(xa), len(xa))].mean() for _ in range(n_boot)])
    db = np.array([xb[rng.integers(0, len(xb), len(xb))].mean() for _ in range(n_boot)])
    d = da - db
    return float(xa.mean() - xb.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

def fmt(ci, pct=False):
    m, lo, hi = ci
    if m != m:
        return "n/a"
    return f"{m*100:5.1f}% [{lo*100:5.1f},{hi*100:5.1f}]" if pct else f"{m:+.3f} [{lo:+.3f},{hi:+.3f}]"

def table(rows, header):
    if not rows:
        print(*header); return
    w = [max(len(str(h)), *(len(str(r[i])) for r in rows)) for i, h in enumerate(header)]
    line = "  ".join(str(h).ljust(w[i]) for i, h in enumerate(header))
    print(line); print("-" * len(line))
    for r in rows:
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(r)))

In [ ]:
# --- 9. Build items: sample, tokenize, length-filter ---
def seeded_sample(rows, n, salt):
    rows = sorted(rows, key=lambda r: r["uid"])
    random.Random(f"{SEED}:{salt}").shuffle(rows)
    return rows[:n], rows[n:]

def finish_item(r):
    r = dict(r)
    r["id"] = f"{r['bench']}:{r['uid']}"
    r["prompt"] = make_prompt(r["question"], r["choices"])
    r["ids"] = encode_user(TOK_R, r["prompt"])           # receiver-template tokens (what the fuser feeds BOTH models)
    r["ids_nat"] = encode_user(TOK_S, r["prompt"])       # sharer's own chat template (sensitivity only)
    r["L"] = len(r["ids"])
    return r

def build_items():
    obqa, _ = seeded_sample(POOLS["obqa"], CFG["N_OBQA"], "obqa")
    arc, _ = seeded_sample(POOLS["arc"], CFG["N_ARC"], "arc")
    mmlu, mmlu_rest = seeded_sample(POOLS["mmlu"], CFG["N_MMLU"], "mmlu")
    reserve = mmlu_rest[:CFG["N_RESERVE_MMLU"]]
    pool = [finish_item(r) for r in mmlu + arc + obqa]
    random.Random(f"{SEED}:order").shuffle(pool)          # pre-registered processing order (stage-wise chunks are unbiased)
    reserve = [finish_item(r) for r in reserve]
    n0 = len(pool)
    pool = [it for it in pool if it["L"] <= CFG["MAX_TOKENS"]]
    reserve = [it for it in reserve if it["L"] <= CFG["MAX_TOKENS"]]
    if len(pool) < n0:
        DEVIATIONS.append(f"dropped {n0-len(pool)} items longer than {CFG['MAX_TOKENS']} tokens (PREREG item filter)")
    return pool, reserve

In [ ]:
# --- 10. Load models (dtype policy + fp16 sanity gate), build items, early timing estimate ---
def probe_condition_outputs(st_items):
    """R / S / C probability vectors on a few items for the dtype gate."""
    out = {"R": [], "S": [], "C": []}
    for it in st_items:
        ids = it["ids"] + RESP_IDS
        for k, lg in (("R", lm_logits(RECV, ids)), ("S", lm_logits(SHAR, ids)), ("C", c2c_logits(ids, ids, it["L"]))):
            p = to_pred(lg)
            out[k].append(p)
    return out

def choose_dtype_and_load():
    want = os.environ.get("PCS_DTYPE") or CFG["DTYPE"]
    table_ = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}
    if want != "auto":
        return build_stack(table_[want]), f"{want} (forced)"
    if SMOKE and not _flag("PCS_SMOKE_GATE"):
        return build_stack(torch.float32), "fp32 (smoke)"
    if NATIVE_BF16 and not SMOKE:
        return build_stack(torch.bfloat16), "bf16 (native on this GPU; same as the C2C repo)"
    # T4/P100: no fast bf16. Try fp16, gate against an fp32 reference on probe items; fall back to fp32.
    probe_n = 24
    st32 = build_stack(torch.float32); use_stack(st32)
    pre_pool = [finish_item(r) for r in seeded_sample(POOLS["obqa"], probe_n, "gate")[0]]
    ref = probe_condition_outputs(pre_pool)
    free_stack(st32)
    report, ok, st16 = {}, True, None
    try:
        st16 = build_stack(torch.float16); use_stack(st16)
        got = probe_condition_outputs(pre_pool)
        for k in ("R", "S", "C"):
            fin = all(g["a"] != "?" for g in got[k])
            agree = float(np.mean([g["a"] == r["a"] for g, r in zip(got[k], ref[k])]))
            dp = float(np.mean([np.abs(np.array(g["p"], float) - np.array(r["p"], float)).mean() for g, r in zip(got[k], ref[k]) if g["a"] != "?"])) if fin else float("nan")
            report[k] = dict(finite=fin, argmax_agree=round(agree, 3), mean_abs_dprob=round(dp, 4))
            ok &= fin and agree >= 0.80 and dp <= 0.10
    except Exception as e:   # any fp16 failure (kernel, dtype mismatch, OOM) -> fp32
        report, ok = {"error": repr(e)[:300]}, False
    print("fp16 gate vs fp32 reference:", report)
    if ok:
        return st16, "fp16 (passed sanity gate vs fp32)"
    if st16 is not None:
        free_stack(st16)
    DEVIATIONS.append(f"fp16 failed the sanity gate {report}; fell back to fp32")
    return build_stack(torch.float32), "fp32 (fp16 failed sanity gate)"

t = time.time()
STACK, DTYPE_NOTE = choose_dtype_and_load()
use_stack(STACK)
COUNTERS["nonfinite"] = 0
print(f"models + fuser ready in {time.time()-t:.0f}s | dtype: {DTYPE_NOTE} | fuser projectors: {STACK['n_proj']}")

POOL, RESERVE = build_items()
print(f"items: pool={len(POOL)} reserve={len(RESERVE)} | median prompt tokens={int(np.median([i['L'] for i in POOL]))} max={max(i['L'] for i in POOL)}")

# bank of same-length partner prompts for the mismatched-sharer control (everything loaded, tokenized lazily by length)
_bank_rows = POOLS["mmlu"] + POOLS["arc"] + POOLS["obqa"]
BANK = {}
for _r in _bank_rows:
    _ids = encode_user(TOK_R, make_prompt(_r["question"], _r["choices"]))
    BANK.setdefault(len(_ids), []).append((f"{_r['bench']}:{_r['uid']}", _ids))
print(f"partner bank: {sum(len(v) for v in BANK.values())} prompts over {len(BANK)} distinct lengths")

def timeit(fn, n=6):
    fn()  # warm-up
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return (time.time() - t0) / n

_it = sorted(POOL, key=lambda i: abs(i["L"] - 250))[0]
_ids = _it["ids"] + RESP_IDS
t_lm = timeit(lambda: lm_logits(RECV, _ids))
t_c2c = timeit(lambda: c2c_logits(_ids, _ids, _it["L"]))
t_gen_r = timeit(lambda: hf_generate(RECV, TOK_R, _it["ids"], GEN_TOKENS, EOS_R, PAD_R), n=2)
t_gen_c = timeit(lambda: c2c_generate(_it["ids"], GEN_TOKENS), n=2)
N_B = len(POOL)
est_stage1 = N_B * (3 * t_lm + t_c2c)
est_stage2 = 0.6 * N_B * (t_c2c + 3 * t_lm + (0.15 if CFG["RUN_T2_RATIONALE"] else 0))
est_C = CFG["N_PLANT"] * (2 * t_c2c + 4 * t_lm)
est_A = CFG["N_PLUMB"] * (3 * t_lm + t_c2c + (2 * t_gen_r + t_gen_c if CFG["RUN_GEN_READOUT_IN_A"] else 0))
print(f"timing (median-length prompt): LM fwd {t_lm*1e3:.0f} ms | C2C fwd {t_c2c*1e3:.0f} ms | HF gen64 {t_gen_r:.2f}s | C2C gen64 {t_gen_c:.2f}s")
print(f"ESTIMATE  Part A ~{est_A/60:.0f} min | Part B ~{(est_stage1+est_stage2)/60:.0f} min | Part C ~{est_C/60:.0f} min  (heuristic; includes no model-download time)")
if (est_stage1 + est_stage2 + est_C) / 60 > CFG["TIME_BUDGET_MIN"] and not (SMOKE or QUICK):
    scale = CFG["TIME_BUDGET_MIN"] * 60 / (est_stage1 + est_stage2 + est_C)
    keep = max(600, int(len(POOL) * scale))
    DEVIATIONS.append(f"estimated B+C runtime exceeded {CFG['TIME_BUDGET_MIN']} min; Part B pool cut from {len(POOL)} to {keep} items (pre-registered order)")
    POOL = POOL[:keep]
    print("!! time budget: Part B pool reduced to", keep)

In [ ]:
# --- 11. Part A: plumbing check against the paper (OpenBookQA, Table 4: receiver 39.2 / sharer 45.6 / C2C 52.6, N=500) ---
PAPER_OBQA = {"R": 39.20, "S": 45.60, "C": 52.60}
RES = {"config": CFG, "deviations": DEVIATIONS, "partA": None, "partB": None, "partC": None}
PLUMBING_OK = None

def run_part_A():
    global PLUMBING_OK
    obqa, _ = seeded_sample(POOLS["obqa"], CFG["N_PLUMB"], "partA")
    items = [finish_item(r) for r in obqa]
    y = np.array([it["y"] for it in items])
    rows = {"gen": {"R": [], "S": [], "C": []}, "logit": {"R": [], "S": [], "C": []}}
    chk = {"norproj_vs_recv_agree": [], "norproj_dprob": [], "list_vs_tensor_maxdiff": []}
    t_a = time.time()
    for n_, it in enumerate(items):
        ids_full, ids_nat_full = it["ids"] + RESP_IDS, it["ids_nat"] + RESP_IDS
        r_lg = lm_logits(RECV, ids_full)
        r_pred = to_pred(r_lg)
        rows["logit"]["R"].append(r_pred["a"])
        rows["logit"]["S"].append(to_pred(lm_logits(SHAR, ids_nat_full))["a"])    # sharer-alone with its own template, as in C2C's baseline
        c_lg = c2c_logits(ids_full, ids_full, it["L"])
        rows["logit"]["C"].append(to_pred(c_lg)["a"])
        off_lg = c2c_logits(ids_full, ids_full, it["L"], sharer_mask=-1)           # projection OFF must reproduce the plain receiver
        off_pred = to_pred(off_lg)
        chk["norproj_vs_recv_agree"].append(float(off_pred["a"] == r_pred["a"]))
        if off_pred["a"] != "?" and r_pred["a"] != "?":
            chk["norproj_dprob"].append(float(np.abs(np.array(off_pred["p"]) - np.array(r_pred["p"])).mean()))
        if n_ < 10:
            chk["list_vs_tensor_maxdiff"].append(float((c2c_logits(ids_full, ids_full, it["L"], as_tensor=True).float() - c_lg.float()).abs().max()))
        if CFG["RUN_GEN_READOUT_IN_A"]:
            rows["gen"]["R"].append(extract_answer_from_content(hf_generate(RECV, TOK_R, it["ids"], GEN_TOKENS, EOS_R, PAD_R)))
            rows["gen"]["S"].append(extract_answer_from_content(hf_generate(SHAR, TOK_S, it["ids_nat"], GEN_TOKENS, EOS_S, PAD_S)))
            rows["gen"]["C"].append(extract_answer_from_content(c2c_generate(it["ids"], GEN_TOKENS)))
        if (n_ + 1) % 50 == 0:
            print(f"  part A {n_+1}/{len(items)}  {time.time()-t_a:.0f}s", flush=True)
    out = {"n": len(items), "paper_obqa_acc": PAPER_OBQA, "readouts": {}, "checks": {}}
    for ro in ("gen", "logit"):
        if not rows[ro]["R"]:
            continue
        corr = {k: np.array([a == b for a, b in zip(rows[ro][k], y)], float) for k in "RSC"}
        out["readouts"][ro] = {
            "acc": {k: fmt(boot_ci(corr[k]), pct=True) for k in "RSC"},
            "acc_point": {k: float(corr[k].mean()) for k in "RSC"},
            "C_minus_R": fmt(boot_ci(corr["C"] - corr["R"])),
            "C_minus_S": fmt(boot_ci(corr["C"] - corr["S"])),
            "C_minus_R_ci": boot_ci(corr["C"] - corr["R"]),
            "unparsed": {k: int(sum(a is None for a in rows[ro][k])) for k in "RSC"},
        }
    out["checks"] = {"projection_off_matches_receiver": float(np.mean(chk["norproj_vs_recv_agree"])),
                     "projection_off_mean_abs_dprob": float(np.mean(chk["norproj_dprob"])) if chk["norproj_dprob"] else None,
                     "list_vs_tensor_max_logit_diff": float(max(chk["list_vs_tensor_maxdiff"])) if chk["list_vs_tensor_maxdiff"] else None,
                     "nonfinite_so_far": COUNTERS["nonfinite"]}
    out["per_item"] = [{"id": it["id"], "y": it["y"], **{f"{ro}_{k}": rows[ro][k][i] for ro in rows for k in "RSC" if rows[ro][k]}} for i, it in enumerate(items)]
    ref_ro = "gen" if "gen" in out["readouts"] else "logit"
    lo = out["readouts"][ref_ro]["C_minus_R_ci"][1]
    out["criteria"] = {"no_nonfinite_logits": COUNTERS["nonfinite"] == 0,
                       f"C2C_beats_receiver_CI_excludes_0[{ref_ro}]": bool(lo > 0),
                       "projection_off_matches_receiver>=95%": bool(out["checks"]["projection_off_matches_receiver"] >= 0.95)}
    PLUMBING_OK = all(out["criteria"].values())
    out["plumbing_ok"] = PLUMBING_OK
    out["seconds"] = round(time.time() - t_a)
    return out

def print_part_A(a):
    print("\n=== PART A: plumbing (OpenBookQA, N=%d; paper N=500: R %.1f / S %.1f / C2C %.1f) ===" % (a["n"], *[PAPER_OBQA[k] for k in "RSC"]))
    rows = []
    for ro, d in a["readouts"].items():
        for k, nm in (("R", "receiver alone"), ("S", "sharer alone"), ("C", "C2C fused")):
            rows.append([ro, nm, d["acc"][k], f"{PAPER_OBQA[k]:.1f}%", d["unparsed"][k]])
    table(rows, ["readout", "condition", "accuracy [95% CI]", "paper", "unparsed"])
    for ro, d in a["readouts"].items():
        print(f"  [{ro}] C2C - receiver: {d['C_minus_R']}   C2C - sharer: {d['C_minus_S']}")
    print("  checks:", a["checks"])
    print("  criteria (PREREG):", a["criteria"])
    print("  PLUMBING:", "OK" if a["plumbing_ok"] else "!! NOT OK: see failed criterion above; Part B is not interpretable until it is understood", f"({a['seconds']}s)")

if CFG["RUN_A"]:
    RES["partA"] = run_part_A()
    print_part_A(RES["partA"])
    if not RES["partA"]["plumbing_ok"]:
        DEVIATIONS.append("Part A plumbing check did NOT pass; treat Parts B/C as uninterpretable")

In [ ]:
# --- 12. Part B: fidelity test (stage 1 on all items, stage 2 on disagreement items) ---
RECORDS = {}   # item id -> {cond: {"a": letter, "p": [4 probs]} | text fields}
ITEMS = {}

def save_partial():
    try:
        PARTIAL_PATH.write_text(json.dumps({"records": RECORDS, "config": CFG}))
    except Exception as e:
        print("could not write partial results:", e)

def stage1(items):
    t_s = time.time()
    for n_, it in enumerate(items):
        ITEMS[it["id"]] = it
        rec = RECORDS.setdefault(it["id"], {})
        if "C" in rec:
            continue
        ids_full, ids_nat_full = it["ids"] + RESP_IDS, it["ids_nat"] + RESP_IDS
        rec["R"] = to_pred(lm_logits(RECV, ids_full))
        rec["S"] = to_pred(lm_logits(SHAR, ids_full))          # sharer fed the exact tokens the fuser feeds it
        rec["S_nat"] = to_pred(lm_logits(SHAR, ids_nat_full))   # sharer with its own chat template (sensitivity)
        rec["C"] = to_pred(c2c_logits(ids_full, ids_full, it["L"]))
        if (n_ + 1) % 100 == 0:
            print(f"  stage1 {n_+1}/{len(items)}  {time.time()-t_s:.0f}s  nonfinite={COUNTERS['nonfinite']}", flush=True)
            save_partial()

def pick_partner(it):
    cands = [(pid, ids) for pid, ids in BANK.get(it["L"], []) if pid != it["id"]]
    if not cands:
        return None
    return random.Random(f"{SEED}:mm:{it['id']}").choice(cands)

def stage2(items):
    """Only items where sharer and receiver disagree: mismatched-sharer control and text handoffs."""
    d_items = [it for it in items if RECORDS[it["id"]]["S"]["a"] != RECORDS[it["id"]]["R"]["a"]
               and "?" not in (RECORDS[it["id"]]["S"]["a"], RECORDS[it["id"]]["R"]["a"])]
    t_s = time.time()
    for n_, it in enumerate(d_items):
        rec = RECORDS[it["id"]]
        s_letter = rec["S"]["a"]
        if "T" not in rec:
            rec["T"] = to_pred(lm_logits(RECV, encode_user(TOK_R, f"Another model answered: {s_letter}\n\n" + it["prompt"]) + RESP_IDS))
        if "C_mm" not in rec:
            part = pick_partner(it)
            if part is None:
                rec["C_mm"] = None
            else:
                ids_full = it["ids"] + RESP_IDS
                rec["C_mm"] = to_pred(c2c_logits(ids_full, part[1] + RESP_IDS, it["L"]))
                rec["C_mm"]["partner"] = part[0]
        if (n_ + 1) % 100 == 0:
            print(f"  stage2 {n_+1}/{len(d_items)}  {time.time()-t_s:.0f}s", flush=True)
            save_partial()
    if CFG["RUN_T2_RATIONALE"]:
        todo = [it for it in d_items if "T2" not in RECORDS[it["id"]]]
        if todo:
            prompts = []
            for it in todo:
                s_letter = RECORDS[it["id"]]["S"]["a"]
                text = TOK_S.apply_chat_template([{"role": "user", "content": it["prompt"]}], tokenize=False, add_generation_prompt=True)
                prompts.append(TOK_S(text + f"The correct answer is {s_letter}. Brief reason:", add_special_tokens=False)["input_ids"])
            gens = hf_generate_batch(SHAR, TOK_S, prompts, 40, EOS_S, PAD_S)
            for it, g in zip(todo, gens):
                why = g.strip().split("\n")[0].strip()
                m = re.search(r"^(.{15,}?[.!?])(\s|$)", why)
                why = (m.group(1) if m else why)[:240] or "(no reason given)"
                s_letter = RECORDS[it["id"]]["S"]["a"]
                RECORDS[it["id"]]["T2_text"] = why
                RECORDS[it["id"]]["T2"] = to_pred(lm_logits(RECV, encode_user(TOK_R, f"Another model answered: {s_letter}. Its reasoning: {why}\n\n" + it["prompt"]) + RESP_IDS))
    save_partial()

def arrays(ids, cond, key="a"):
    return np.array([RECORDS[i][cond][key] if RECORDS[i].get(cond) else "" for i in ids])

def analyze_B(ids):
    ids = [i for i in ids if all(k in RECORDS[i] for k in ("R", "S", "S_nat", "C"))]
    y = np.array([ITEMS[i]["y"] for i in ids])
    R, S, Sn, C = (arrays(ids, k) for k in ("R", "S", "S_nat", "C"))
    ok = (R != "?") & (S != "?") & (C != "?")
    D = ok & (S != R)
    cells = {"A": D & (S == y), "B": D & (S != y) & (R == y), "W": D & (S != y) & (R != y)}
    cells["B+W"] = cells["B"] | cells["W"]
    has = lambda cond: np.array([bool(RECORDS[i].get(cond)) and RECORDS[i][cond]["a"] != "?" for i in ids])
    C_mm, T, T2 = arrays(ids, "C_mm"), arrays(ids, "T"), arrays(ids, "T2")
    hm, ht, ht2 = has("C_mm"), has("T"), has("T2")
    out = {"n_items": len(ids), "n_disagree": int(D.sum()), "cell_sizes": {k: int(v.sum()) for k, v in cells.items()},
           "mismatch_partner_coverage_on_D": float(hm[D].mean()) if D.any() else None}
    out["accuracy_all"] = {k: fmt(boot_ci((a == y)[ok]), pct=True) for k, a in (("R", R), ("S", S), ("S_nat", Sn), ("C", C))}
    out["accuracy_all_point"] = {k: float((a == y)[ok].mean()) for k, a in (("R", R), ("S", S), ("S_nat", Sn), ("C", C))}
    out["accuracy_all"]["C_minus_R"] = fmt(boot_ci(((C == y).astype(float) - (R == y).astype(float))[ok]))
    out["accuracy_on_D"] = {k: fmt(boot_ci((a == y)[D & h]), pct=True) for k, a, h in
                            (("R", R, np.ones_like(D)), ("S", S, np.ones_like(D)), ("C", C, np.ones_like(D)),
                             ("C_mm", C_mm, hm), ("T", T, ht), ("T2", T2, ht2))}
    out["accuracy_on_D"]["C_minus_C_mm"] = fmt(boot_ci(((C == y).astype(float) - (C_mm == y).astype(float))[D & hm]))
    out["accuracy_on_D"]["C_minus_C_mm_ci"] = boot_ci(((C == y).astype(float) - (C_mm == y).astype(float))[D & hm])
    inh = {}
    for cn, cm in cells.items():
        row = {"n": int(cm.sum())}
        row["Inh_C"] = boot_ci((C == S)[cm]); row["Rev_C"] = boot_ci((C == R)[cm])
        row["Inh_C_other"] = boot_ci(((C != S) & (C != R))[cm])
        mm = cm & hm
        row["n_with_mm"] = int(mm.sum())
        row["Inh_C_mm"] = boot_ci((C_mm == S)[mm]); row["Inh_C_same_items"] = boot_ci((C == S)[mm])
        row["Rev_C_mm"] = boot_ci((C_mm == R)[mm]); row["Rev_C_same_items"] = boot_ci((C == R)[mm])
        row["E_excess"] = boot_ci(((C == S).astype(float) - (C_mm == S).astype(float))[mm])
        row["Inh_T"] = boot_ci((T == S)[cm & ht]); row["Inh_T2"] = boot_ci((T2 == S)[cm & ht2])
        row["Inh_C_native_S"] = boot_ci((C == Sn)[cm & (Sn != "?")])
        inh[cn] = row
    out["inheritance"] = inh
    out["asymmetry_Inh_C_A_minus_B"] = boot_ci_unpaired_diff((C == S)[cells["A"]], (C == S)[cells["B"]])
    eB = inh["B"]["E_excess"]; tB = inh["B"]["Inh_T"]
    out["bridge_over_text_B"] = (eB[0] / tB[0]) if tB[0] and tB[0] > 0 else None
    out["S_native_agreement"] = float((S == Sn)[ok].mean())
    out["letter_marginals"] = {k: {l: float((a == l).mean()) for l in LETTERS} for k, a in (("R", R), ("S", S), ("C", C))}
    mm_leak = [(RECORDS[i]["C_mm"]["a"], RECORDS[RECORDS[i]["C_mm"]["partner"]]["S"]["a"]) for i in ids
               if RECORDS[i].get("C_mm") and RECORDS[i]["C_mm"].get("partner") in RECORDS and "S" in RECORDS[RECORDS[i]["C_mm"]["partner"]]]
    out["mismatch_letter_leak_vs_partner_sharer_answer"] = {"n": len(mm_leak), "rate": float(np.mean([a == b for a, b in mm_leak])) if mm_leak else None, "chance": 0.25}
    out["reading"] = auto_reading(out)
    return out

def auto_reading(o):
    B_, A_ = o["inheritance"]["B"], o["inheritance"]["A"]
    eB, eA = B_["E_excess"], A_["E_excess"]
    if o["cell_sizes"]["A"] < CFG["MIN_CELL"] or o["cell_sizes"]["B"] < CFG["MIN_CELL"]:
        return f"UNDERPOWERED: |A|={o['cell_sizes']['A']} |B|={o['cell_sizes']['B']} (< {CFG['MIN_CELL']}); no claim (PREREG)"
    if eB[1] >= 0.10:
        return "BELIEF TRANSFER: receiver abandons a correct answer for the sharer's wrong one, >=10pp beyond the mismatched-sharer null (lower CI of E_B >= 0.10)"
    if eB[2] < 0.10 and eA[0] >= 0.10:
        return "TRUTH-BIASED HELPER: follows the sharer when it is right (E_A >= 0.10) but not when it is wrong (upper CI of E_B < 0.10)"
    d = o["accuracy_on_D"]["C_minus_C_mm_ci"]
    if eA[1] >= -0.05 and eA[2] <= 0.05 and eB[1] >= -0.05 and eB[2] <= 0.05 and d[1] <= 0 <= d[2]:
        return "NON-SPECIFIC PERTURBATION: inheritance is indistinguishable from the mismatched-sharer null and accuracy does not depend on the sharer's content"
    return "MIXED / UNDERPOWERED: matches no pre-registered row; no claim"

def print_part_B(b):
    print("\n=== PART B: fidelity (logit readout; D = items where sharer-alone != receiver-alone) ===")
    print(f"items scored: {b['n_items']} | disagreement set D: {b['n_disagree']} | cells: {b['cell_sizes']} | sharer own-template vs fuser-tokens agreement: {b['S_native_agreement']:.2f}")
    print(f"same-length partner found for the mismatched-sharer control on {100*(b['mismatch_partner_coverage_on_D'] or 0):.0f}% of D items")
    table([[k, v] for k, v in b["accuracy_all"].items() if k != "C_minus_R"] + [["C - R (paired)", b["accuracy_all"]["C_minus_R"]]], ["accuracy, all items", "[95% CI]"])
    print()
    table([[k, v] for k, v in b["accuracy_on_D"].items() if not k.endswith("_ci")], ["accuracy on D", "[95% CI]"])
    print("\nInheritance P(answer == sharer's answer). A: sharer right/receiver wrong | B: sharer WRONG/receiver right | W: both wrong, differ")
    rows = []
    for cn in ("A", "B", "W", "B+W"):
        r = b["inheritance"][cn]
        rows.append([cn, r["n"], fmt(r["Inh_C"], True), fmt(r["Rev_C"], True), fmt(r["Inh_C_mm"], True), fmt(r["Rev_C_mm"], True), fmt(r["E_excess"]), fmt(r["Inh_T"], True), fmt(r["Inh_T2"], True) if CFG["RUN_T2_RATIONALE"] else "-"])
    table(rows, ["cell", "n", "C2C inherits S", "C2C reverts to R", "null inherits S", "null reverts to R", "EXCESS E = C2C - null", "text handoff", "text+rationale"])
    print("(null = C2C with the sharer cache computed on a different same-length prompt. 'reverts' columns show the two perturbations are comparable in size.)")
    print(f"\nasymmetry Inh(C|A) - Inh(C|B): {fmt(b['asymmetry_Inh_C_A_minus_B'])}    bridge/text ratio in B: {b['bridge_over_text_B'] if b['bridge_over_text_B'] is None else round(b['bridge_over_text_B'], 2)}")
    print("predicted-letter marginals:", {k: {l: round(x, 2) for l, x in v.items()} for k, v in b["letter_marginals"].items()})
    print("mismatched-sharer letter leak (C_mm answer == partner's sharer answer):", b["mismatch_letter_leak_vs_partner_sharer_answer"])
    print("\nAUTO-READING (PREREG decision table):", b["reading"])
    print(f"non-finite logits so far: {COUNTERS['nonfinite']}")

def run_part_B():
    t_b = time.time()
    stage1(POOL); stage2(POOL); done = list(POOL)
    b = analyze_B([i["id"] for i in done])
    reserve = list(RESERVE)
    while reserve and (b["cell_sizes"]["A"] < CFG["MIN_CELL"] or b["cell_sizes"]["B"] < CFG["MIN_CELL"]):
        chunk, reserve = reserve[:400], reserve[400:]
        DEVIATIONS.append(f"extended Part B with {len(chunk)} reserve MMLU-Redux items because a cell had <{CFG['MIN_CELL']} items (PREREG allows this)")
        stage1(chunk); stage2(chunk); done += chunk
        b = analyze_B([i["id"] for i in done])
    b["seconds"] = round(time.time() - t_b)
    b["nonfinite_logits"] = COUNTERS["nonfinite"]
    return b, done

if CFG["RUN_B"]:
    RES["partB"], DONE_B = run_part_B()
    print_part_B(RES["partB"])

In [ ]:
# --- 13. Part C: planted belief (sharer sees a false premise; receiver sees masked padding of equal length) ---
def build_plant(it, w):
    note = lambda L: f"Reference note: the correct answer to the question below is {L}.\n\n"
    plain = it["ids"]
    ids_w = encode_user(TOK_R, note(w) + it["prompt"])
    ids_x = encode_user(TOK_R, note("X") + it["prompt"])
    k = len(ids_w) - len(plain)
    if k <= 0 or len(ids_x) != len(ids_w):
        return None
    p = next((j for j in range(len(plain)) if plain[j] != ids_w[j]), len(plain))
    if ids_w[p + k:] != plain[p:] or ids_x[:p] != ids_w[:p] or ids_x[p + k:] != ids_w[p + k:]:
        return None
    n = len(ids_w) + len(RESP_IDS)
    mask_r = [1] * p + [0] * k + [1] * (n - p - k)
    ids_r = ids_w[:p] + [PAD_R] * k + ids_w[p + k:] + RESP_IDS     # receiver: premise replaced by masked padding
    return dict(ids_w=ids_w + RESP_IDS, ids_x=ids_x + RESP_IDS, ids_r=ids_r, mask_r=mask_r, L=len(ids_w), p=p, k=k)

def run_part_C():
    t_c = time.time()
    cand = [it for it in POOL if it["id"] in RECORDS and "R" in RECORDS[it["id"]]][: CFG["N_PLANT"] * 2]
    rows, skipped = [], 0
    for it in cand:
        if len(rows) >= CFG["N_PLANT"]:
            break
        w = random.Random(f"{SEED}:plant:{it['id']}").choice([l for l in LETTERS if l != it["y"]])
        b = build_plant(it, w)
        if b is None:
            skipped += 1
            continue
        c_plant = to_pred(c2c_logits(b["ids_r"], b["ids_w"], b["L"], mask_r=b["mask_r"]))
        c_ctrl = to_pred(c2c_logits(b["ids_r"], b["ids_x"], b["L"], mask_r=b["mask_r"]))
        s_plant, s_ctrl = to_pred(lm_logits(SHAR, b["ids_w"])), to_pred(lm_logits(SHAR, b["ids_x"]))
        t_plant = to_pred(lm_logits(RECV, b["ids_w"]))                                   # receiver is simply told, in text
        r_pad = to_pred(lm_logits(RECV, b["ids_r"], mask=b["mask_r"]))                  # what the receiver sees inside C_plant / C_ctrl, bridge off
        plain_full = it["ids"] + RESP_IDS                                                 # masking sanity: same tokens as plain, positions skip the masked gap,
        r_gap = to_pred(lm_logits(RECV, plain_full, positions=[j if j < b["p"] else j + b["k"] for j in range(len(plain_full))]))   # so R_pad must equal R_gap exactly (up to float noise)
        rows.append(dict(id=it["id"], y=it["y"], w=w, C_plant=c_plant, C_ctrl=c_ctrl, S_plant=s_plant, S_ctrl=s_ctrl,
                         T_plant=t_plant, R_pad=r_pad, R_gap=r_gap, R=RECORDS[it["id"]]["R"]))
        if len(rows) % 100 == 0:
            print(f"  part C {len(rows)}/{CFG['N_PLANT']}  {time.time()-t_c:.0f}s", flush=True)
    a = lambda k: np.array([r[k]["a"] for r in rows])
    w = np.array([r["w"] for r in rows])
    ok = np.ones(len(rows), bool)
    for k in ("C_plant", "C_ctrl", "S_plant", "S_ctrl", "T_plant", "R", "R_pad", "R_gap"):
        ok &= a(k) != "?"
    hit = lambda k: (a(k) == w).astype(float)
    susc = ok & (a("S_plant") == w) & (a("S_ctrl") != w)
    out = {"n": int(ok.sum()), "n_skipped_construction": skipped, "n_sharer_susceptible": int(susc.sum()),
           "P_follow": {k: fmt(boot_ci(hit(k)[ok]), True) for k in ("S_plant", "S_ctrl", "C_plant", "C_ctrl", "T_plant", "R")},
           "sharer_effect_S_plant_minus_S_ctrl": fmt(boot_ci((hit("S_plant") - hit("S_ctrl"))[ok])),
           "bridge_effect_all": fmt(boot_ci((hit("C_plant") - hit("C_ctrl"))[ok])),
           "bridge_effect_susceptible": fmt(boot_ci((hit("C_plant") - hit("C_ctrl"))[susc])),
           "bridge_effect_susceptible_ci": boot_ci((hit("C_plant") - hit("C_ctrl"))[susc]),
           "text_effect_T_plant_minus_R": fmt(boot_ci((hit("T_plant") - hit("R"))[ok])),
           "mask_sanity_same_answer": float((a("R_pad") == a("R_gap"))[ok].mean()) if ok.any() else None,
           "mask_sanity_mean_abs_dprob": float(np.mean([np.abs(np.array(r["R_pad"]["p"], float) - np.array(r["R_gap"]["p"], float)).mean() for r, k_ in zip(rows, ok) if k_])) if ok.any() else None,
           "receiver_pad_vs_plain_same_answer": float((a("R_pad") == a("R"))[ok].mean()) if ok.any() else None,
           "seconds": round(time.time() - t_c), "per_item": rows}
    lo = out["bridge_effect_susceptible_ci"][1]
    out["reading"] = ("BRIDGE CARRIES SHARER-PRIVATE INFORMATION (lower CI >= 0.05)" if lo >= 0.05 else
                      "no detectable transfer of the planted premise (exploratory; fuser was only trained on identical inputs)" if susc.sum() >= 30 else
                      "too few items where the sharer itself adopted the false premise; no claim")
    return out

def print_part_C(c):
    print("\n=== PART C: planted false belief (exploratory) ===")
    print(f"items: {c['n']} (skipped {c['n_skipped_construction']}) | sharer adopted the planted letter on {c['n_sharer_susceptible']}")
    table([[k, v] for k, v in c["P_follow"].items()], ["P(answer == planted letter w)", "[95% CI]"])
    print("sharer effect (S_plant - S_ctrl):      ", c["sharer_effect_S_plant_minus_S_ctrl"])
    print("bridge effect, all items:              ", c["bridge_effect_all"])
    print("bridge effect, sharer-susceptible items:", c["bridge_effect_susceptible"])
    print("text effect (receiver told directly):  ", c["text_effect_T_plant_minus_R"])
    print("mask sanity (masked-pad receiver vs plain receiver with the same position gap; should be ~identical):",
          "same answer", c["mask_sanity_same_answer"], "| mean |dprob|", c["mask_sanity_mean_abs_dprob"])
    print("   (receiver w/ masked pad vs plain receiver: same answer", c["receiver_pad_vs_plain_same_answer"], "; differs only by the positional gap)")
    print("READING:", c["reading"])

if CFG["RUN_C"] and CFG["RUN_B"]:
    RES["partC"] = run_part_C()
    print_part_C(RES["partC"])

In [ ]:
# --- 14. Save everything and print the final summary ---
def jsonable(o):
    if isinstance(o, (np.floating, np.integer, np.bool_)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, tuple):
        return list(o)
    raise TypeError(type(o))

if CFG["RUN_B"]:
    for it_id, rec in RECORDS.items():
        it = ITEMS[it_id]
        rec["_meta"] = {"bench": it["bench"], "y": it["y"], "L": it["L"], "subject": it.get("subject")}
RES["run"] = {"date_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "gpu": GPU_NAME, "dtype": str(DTYPE_USED), "dtype_note": DTYPE_NOTE,
              "torch": torch.__version__, "transformers": installed("transformers"), "c2c_commit": head,
              "receiver": RECV_ID, "sharer": SHAR_ID, "fuser": f"{FUSER_REPO}/{FUSER_SUB}/final", "smoke": SMOKE, "quick": QUICK,
              "wall_seconds": round(time.time() - T0), "nonfinite_logits": COUNTERS["nonfinite"], "plumbing_ok": PLUMBING_OK}
RES["records"] = RECORDS if CFG["RUN_B"] else None
if RES["partB"]:
    _b, _inh = RES["partB"], RES["partB"]["inheritance"]
    RES["summary"] = {
        "plumbing_ok": PLUMBING_OK,
        "cell_sizes": _b["cell_sizes"],
        "headline_cell_B_sharer_wrong_receiver_right": {
            "C2C_inherits_sharer_answer": _inh["B"]["Inh_C"], "mismatched_sharer_null": _inh["B"]["Inh_C_mm"],
            "excess_E": _inh["B"]["E_excess"], "text_handoff": _inh["B"]["Inh_T"], "text_plus_rationale": _inh["B"]["Inh_T2"]},
        "cell_A_sharer_right_receiver_wrong_excess_E": _inh["A"]["E_excess"],
        "asymmetry_Inh_C_A_minus_B": _b["asymmetry_Inh_C_A_minus_B"],
        "accuracy_all": _b["accuracy_all"], "partB_reading": _b["reading"],
        "partC_bridge_effect_susceptible": RES["partC"]["bridge_effect_susceptible"] if RES["partC"] else None,
        "partC_reading": RES["partC"]["reading"] if RES["partC"] else None,
    }
RESULTS_PATH.write_text(json.dumps(RES, default=jsonable))
print(f"\nsaved {RESULTS_PATH} ({RESULTS_PATH.stat().st_size/1e6:.1f} MB). Total wall time {(time.time()-T0)/60:.1f} min.")
if RES.get("summary"):
    h = RES["summary"]["headline_cell_B_sharer_wrong_receiver_right"]
    print("\n" + "=" * 78)
    print("HEADLINE (cell B: sharer WRONG, receiver right; n=%d)" % RES["summary"]["cell_sizes"]["B"])
    print("  C2C inherits the sharer's wrong answer :", fmt(h["C2C_inherits_sharer_answer"], True))
    print("  mismatched-sharer null                 :", fmt(h["mismatched_sharer_null"], True))
    print("  EXCESS inheritance E (C2C - null)      :", fmt(h["excess_E"]))
    print("  text handoff 'Another model answered'  :", fmt(h["text_handoff"], True))
    print("  PartB reading:", RES["summary"]["partB_reading"])
    if RES["summary"]["partC_reading"]:
        print("  PartC reading:", RES["summary"]["partC_reading"])
    print("  plumbing:", "OK" if PLUMBING_OK else "NOT OK / not run")
    print("=" * 78)
if DEVIATIONS:
    print("DEVIATIONS from PREREG:", *DEVIATIONS, sep="\n  - ")
if SMOKE or QUICK:
    print("NOTE: this was a", "SMOKE (random tiny models)" if SMOKE else "QUICK", "run. Do not interpret numbers.")